# Reporting and API Contract Walkthrough

This notebook shows the read/reporting side of the repo and the thin API
contract layer that now sits above the service responses.


## Setup

The setup cell seeds a small collection with one intentionally mismatched
price/finish case so the reporting and reconcile flows have something real
to show.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mtg_source_stack').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

WORK_DIR = REPO_ROOT / 'var' / 'notebooks' / '04_reporting_and_api_contract'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

from mtg_source_stack.api_contract import api_error_payload, api_error_status
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.errors import ConflictError, NotFoundError, SchemaNotReadyError, ValidationError
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards
from mtg_source_stack.inventory.report_formatters import (
    format_export_csv_result,
    format_inventory_report_result,
    format_owned_rows,
    format_reconcile_prices_result,
)
from mtg_source_stack.inventory.report_helpers import render_table
from mtg_source_stack.inventory.response_models import serialize_response
from mtg_source_stack.inventory.service import (
    add_card,
    create_inventory,
    export_inventory_csv,
    inventory_report,
    list_owned_filtered,
    list_price_gaps,
    reconcile_prices,
    valuation_filtered,
)

DB_PATH = WORK_DIR / 'reporting_walkthrough.sqlite3'
SCRYFALL_JSON = WORK_DIR / 'scryfall_sample.json'
IDENTIFIERS_JSON = WORK_DIR / 'identifiers_sample.json'
PRICES_JSON = WORK_DIR / 'prices_sample.json'
EXPORT_CSV = WORK_DIR / 'burn_rows.csv'


def write_sample_payloads() -> None:
    scryfall_payload = [
        {
            'id': 's1',
            'oracle_id': 'o1',
            'name': 'Lightning Bolt',
            'set': 'lea',
            'set_name': 'Limited Edition Alpha',
            'collector_number': '161',
            'lang': 'en',
            'rarity': 'common',
            'released_at': '1993-08-05',
            'mana_cost': '{R}',
            'type_line': 'Instant',
            'oracle_text': 'Lightning Bolt deals 3 damage to any target.',
            'colors': ['R'],
            'color_identity': ['R'],
            'finishes': ['nonfoil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/lightning-bolt'},
            'tcgplayer_id': 534658,
        },
        {
            'id': 's2',
            'oracle_id': 'o2',
            'name': 'Sol Ring',
            'set': 'clb',
            'set_name': "Commander Legends: Battle for Baldur's Gate",
            'collector_number': '860',
            'lang': 'en',
            'rarity': 'rare',
            'released_at': '2022-06-10',
            'mana_cost': '{1}',
            'type_line': 'Artifact',
            'oracle_text': '{T}: Add {C}{C}.',
            'colors': [],
            'color_identity': [],
            'finishes': ['nonfoil', 'foil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/sol-ring'},
            'tcgplayer_id': 271111,
        },
        {
            'id': 's3',
            'oracle_id': 'o3',
            'name': 'Counterspell',
            'set': '7ed',
            'set_name': 'Seventh Edition',
            'collector_number': '67',
            'lang': 'en',
            'rarity': 'uncommon',
            'released_at': '2001-04-11',
            'mana_cost': '{U}{U}',
            'type_line': 'Instant',
            'oracle_text': 'Counter target spell.',
            'colors': ['U'],
            'color_identity': ['U'],
            'finishes': ['nonfoil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/counterspell'},
            'tcgplayer_id': 333333,
        },
        {
            'id': 's4',
            'oracle_id': 'o4',
            'name': 'Shiny Bird',
            'set': 'neo',
            'set_name': 'Kamigawa: Neon Dynasty',
            'collector_number': '12',
            'lang': 'en',
            'rarity': 'rare',
            'released_at': '2022-02-18',
            'mana_cost': '{2}{W}',
            'type_line': 'Creature - Bird',
            'oracle_text': 'Flying',
            'colors': ['W'],
            'color_identity': ['W'],
            'finishes': ['nonfoil', 'foil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/shiny-bird'},
            'tcgplayer_id': 444444,
        },
    ]

    identifiers_payload = {
        'data': {
            'uuid-1': {'identifiers': {'scryfallId': 's1', 'tcgplayerProductId': '534658'}},
            'uuid-2': {'identifiers': {'scryfallId': 's2', 'tcgplayerProductId': '271111'}},
            'uuid-3': {'identifiers': {'scryfallId': 's3', 'tcgplayerProductId': '333333'}},
            'uuid-4': {'identifiers': {'scryfallId': 's4', 'tcgplayerProductId': '444444'}},
        }
    }

    prices_payload = {
        'data': {
            'uuid-1': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 2.92}}}}},
            'uuid-2': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 1.75}, 'foil': {'2026-03-27': 4.25}}}}},
            'uuid-3': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 1.25}}}}},
            'uuid-4': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'foil': {'2026-03-27': 6.50}}}}},
        }
    }

    SCRYFALL_JSON.write_text(json.dumps(scryfall_payload), encoding='utf-8')
    IDENTIFIERS_JSON.write_text(json.dumps(identifiers_payload), encoding='utf-8')
    PRICES_JSON.write_text(json.dumps(prices_payload), encoding='utf-8')


def seed_demo_inventory() -> None:
    initialize_database(DB_PATH)
    write_sample_payloads()
    import_scryfall_cards(DB_PATH, SCRYFALL_JSON)
    import_mtgjson_identifiers(DB_PATH, IDENTIFIERS_JSON)
    import_mtgjson_prices(DB_PATH, PRICES_JSON)
    create_inventory(DB_PATH, slug='personal', display_name='Personal Collection', description='Demo notebook inventory')

    add_card(
        DB_PATH,
        inventory_slug='personal',
        inventory_display_name=None,
        scryfall_id='s1',
        tcgplayer_product_id=None,
        name=None,
        set_code=None,
        collector_number=None,
        lang=None,
        quantity=3,
        condition_code='NM',
        finish='normal',
        language_code='en',
        location='Red Binder',
        acquisition_price='2.00',
        acquisition_currency='USD',
        notes='Playset',
        tags='burn,trade',
    )
    add_card(
        DB_PATH,
        inventory_slug='personal',
        inventory_display_name=None,
        scryfall_id='s2',
        tcgplayer_product_id=None,
        name=None,
        set_code=None,
        collector_number=None,
        lang=None,
        quantity=1,
        condition_code='NM',
        finish='foil',
        language_code='en',
        location='Commander Deck Box',
        acquisition_price='4.00',
        acquisition_currency='USD',
        notes='Main deck copy',
        tags='commander',
    )
    add_card(
        DB_PATH,
        inventory_slug='personal',
        inventory_display_name=None,
        scryfall_id='s3',
        tcgplayer_product_id=None,
        name=None,
        set_code=None,
        collector_number=None,
        lang=None,
        quantity=2,
        condition_code='NM',
        finish='normal',
        language_code='en',
        location='',
        acquisition_price=None,
        acquisition_currency=None,
        notes=None,
        tags=None,
    )
    add_card(
        DB_PATH,
        inventory_slug='personal',
        inventory_display_name=None,
        scryfall_id='s4',
        tcgplayer_product_id=None,
        name=None,
        set_code=None,
        collector_number=None,
        lang=None,
        quantity=1,
        condition_code='NM',
        finish='normal',
        language_code='en',
        location='White Binder',
        acquisition_price=None,
        acquisition_currency=None,
        notes='Finish intentionally mismatched for reporting demo',
        tags='spec',
    )


print('repo root:', REPO_ROOT)
print('work dir:', WORK_DIR)
print('db path:', DB_PATH)


## Build a Small Demo Collection

The seeded rows intentionally cover three common reporting cases:

- normal valuation (`Lightning Bolt`, `Sol Ring`)
- missing metadata (`Counterspell`)
- a finish that lacks a current matching price (`Shiny Bird`)


In [ ]:
seed_demo_inventory()

owned_rows = serialize_response(
    list_owned_filtered(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        limit=None,
        query=None,
        set_code=None,
        rarity=None,
        finish=None,
        condition_code=None,
        language_code=None,
        location=None,
        tags=None,
    )
)
valuation_rows = serialize_response(
    valuation_filtered(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        query=None,
        set_code=None,
        rarity=None,
        finish=None,
        condition_code=None,
        language_code=None,
        location=None,
        tags=None,
    )
)

print('Owned rows')
print(format_owned_rows(owned_rows))
print()
print('Valuation totals')
print(render_table(valuation_rows, [
    ('provider', 'provider'),
    ('currency', 'currency'),
    ('item_rows', 'item_rows'),
    ('total_cards', 'total_cards'),
    ('total_value', 'total_value'),
]))


## Price Gaps, Reconcile Suggestions, and the Inventory Report

`reconcile_prices()` is suggestion-only. It does not rewrite finish anymore;
it simply shows rows where the latest current price data disagrees with the
stored finish.


In [ ]:
gap_rows = serialize_response(
    list_price_gaps(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        limit=None,
    )
)
reconcile_result = serialize_response(
    reconcile_prices(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        apply_changes=False,
    )
)
report_result = serialize_response(
    inventory_report(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        query=None,
        set_code=None,
        rarity=None,
        finish=None,
        condition_code=None,
        language_code=None,
        location=None,
        tags=None,
        limit=10,
        stale_days=30,
    )
)

print('Price gaps')
print(render_table(gap_rows, [
    ('item_id', 'item_id'),
    ('card_name', 'name'),
    ('finish', 'finish'),
    ('available_finishes', 'available_finishes'),
    ('reconcile_status', 'status'),
]))
print()
print(format_reconcile_prices_result(reconcile_result))
print()
print(format_inventory_report_result(report_result))


## Export a Filtered CSV

The export helpers sit on top of the same filtered list/valuation paths that
power the CLI reports, so they are a good way to inspect the reporting stack
end to end.


In [ ]:
export_result = serialize_response(
    export_inventory_csv(
        DB_PATH,
        inventory_slug='personal',
        provider='tcgplayer',
        output_path=EXPORT_CSV,
        query=None,
        set_code=None,
        rarity=None,
        finish=None,
        condition_code=None,
        language_code=None,
        location=None,
        tags=['burn'],
        limit=None,
    )
)

print(format_export_csv_result(export_result))
print()
print('\n'.join(EXPORT_CSV.read_text(encoding='utf-8').splitlines()[:5]))


## API Contract Examples

The API contract layer is intentionally small. It freezes two things that a
frontend needs to trust:

- typed service data serializes cleanly to JSON
- domain errors map to stable HTTP-style status codes and payloads


In [ ]:
error_examples = [
    ValidationError('Finish must be one of: normal, nonfoil, foil, etched.'),
    NotFoundError('Inventory row was not found.'),
    ConflictError('Changing location would collide with an existing inventory row.'),
    SchemaNotReadyError('Database schema is not current. Run migrate-db before reading from this database.'),
]

error_rows = []
for exc in error_examples:
    payload = api_error_payload(exc)
    error_rows.append(
        {
            'exception': type(exc).__name__,
            'status': api_error_status(exc),
            'code': payload['error']['code'],
            'message': payload['error']['message'],
        }
    )

sample_json = {
    'owned_row': owned_rows[0],
    'valuation_row': valuation_rows[0],
}

print(render_table(error_rows, [('exception', 'exception'), ('status', 'status'), ('code', 'code'), ('message', 'message')]))
print()
print(json.dumps(sample_json, indent=2))
